## 1. Setup & Data Loading

In [1]:
!pip install autogluon


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# AutoGluon
from autogluon.tabular import TabularDataset, TabularPredictor

In [3]:
DATA_PATH = Path("/home/jovyan/__DATA/APBDID_F25/data/handm")

In [4]:
# Load datasets
articles_df = pd.read_csv(DATA_PATH / "articles.csv")
customers_df = pd.read_csv(DATA_PATH / "customers.csv")
transactions_df = pd.read_csv(DATA_PATH / "transactions_train.csv")

print(f"Articles: {articles_df.shape}")
print(f"Customers: {customers_df.shape}")
print(f"Transactions: {transactions_df.shape}")

Articles: (105542, 25)
Customers: (1371980, 7)
Transactions: (31788324, 5)


In [5]:
# Quick look at transactions (our main table)
transactions_df.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


## 2. Minimal Preprocessing

Keep it simple:
- Sample transactions (full dataset is too large for quick experiments)
- Merge customer and article features
- Target: `article_id` (what article was purchased)

In [6]:
# Sample for speed - use more data for better results
SAMPLE_SIZE = 50_000
np.random.seed(42)

transactions_sample = transactions_df.sample(n=SAMPLE_SIZE, random_state=42)
print(f"Sampled transactions: {len(transactions_sample):,}")

Sampled transactions: 50,000


In [7]:
# Select ONLY the target from articles (no other article features to avoid leakage)
articles_target = articles_df[["article_id", "product_group_name"]].copy()

# Select key features from customers (these are legitimate predictors)
customers_features = customers_df[[
    "customer_id",
    "club_member_status",
    "fashion_news_frequency",
    "age"
]].copy()

# Fill missing values
customers_features["club_member_status"] = customers_features["club_member_status"].fillna("UNKNOWN")
customers_features["fashion_news_frequency"] = customers_features["fashion_news_frequency"].fillna("UNKNOWN")
customers_features["age"] = customers_features["age"].fillna(customers_features["age"].median())

In [8]:
# Merge: transactions + customer features + target only
train_df = transactions_sample.merge(customers_features, on="customer_id", how="left")
train_df = train_df.merge(articles_target, on="article_id", how="left")

print(f"Training data shape: {train_df.shape}")
train_df.head()

Training data shape: (50000, 9)


,t_dat,customer_id,article_id,price,sales_channel_id,club_member_status,fashion_news_frequency,age,product_group_name
0,2019-09-13,215895f90002eb3d1a04bd603513c8e85e6002ef08f136...,786586001,0.022017,1,ACTIVE,NONE,41.0,Garment Lower body
1,2019-02-23,7b183268e3a4623b80d5325ec4a20a0af0edff7bcb1748...,658911001,0.028797,2,ACTIVE,Regularly,30.0,Nightwear
2,2019-07-17,2eb7412239a90c0570cd3d1bf0492856ae5b59058b1ea6...,759326005,0.050831,2,ACTIVE,Regularly,22.0,Swimwear
3,2019-05-16,74f162e5a170fd57207aa2a7d5c58479ee9de903b2a277...,737137004,0.027102,1,ACTIVE,NONE,21.0,Garment Upper body
4,2019-08-10,aab9306ee28c4db494003955f80355e540b01480ab35cf...,785931001,0.050831,2,ACTIVE,Regularly,49.0,Garment Lower body


In [9]:
# Simplify target: predict product_group_name instead of exact article_id
# (article_id has too many unique values for quick training)
TARGET = "product_group_name"

print(f"Target classes: {train_df[TARGET].nunique()}")
train_df[TARGET].value_counts()

Target classes: 15


product_group_name
Garment Upper body       19801
Garment Lower body       11086
Garment Full body         5502
Underwear                 4036
Swimwear                  4032
Accessories               2577
Shoes                     1147
Socks & Tights            1060
Nightwear                  585
Unknown                    159
Items                        7
Bags                         4
Cosmetic                     2
Garment and Shoe care        1
Furniture                    1
Name: count, dtype: int64

In [10]:
# Features for training (drop IDs and non-predictive columns)
drop_cols = ["customer_id", "article_id", "t_dat"]
feature_cols = [c for c in train_df.columns if c not in drop_cols and c != TARGET]

train_data = train_df[feature_cols + [TARGET]].copy()
print(f"Features: {feature_cols}")
print(f"Final training shape: {train_data.shape}")

Features: ['price', 'sales_channel_id', 'club_member_status', 'fashion_news_frequency', 'age']
Final training shape: (50000, 6)


## 3. Train AutoGluon Model

AutoGluon handles:
- Automatic feature engineering
- Model selection & ensembling
- Hyperparameter tuning

In [11]:
# Convert to AutoGluon dataset
train_ag = TabularDataset(train_data)

In [12]:
# Train model with time limit (quick baseline)
predictor = TabularPredictor(
    label=TARGET,
    eval_metric="accuracy",
    path="models/autogluon_baseline"
).fit(
    train_ag,
    time_limit=10*60,  # 10 minutes for quick demo
    presets="medium_quality"  # Options: best_quality, high_quality, medium_quality, optimize_for_deployment
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.7
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #216-Ubuntu SMP Thu Aug 29 13:26:53 UTC 2024
CPU Count:          2
Memory Avail:       21.60 GB / 31.35 GB (68.9%)
Disk Space Avail:   4.56 GB / 11.71 GB (39.0%)
	We recommend a minimum available disk space of 10 GB, and large datasets may require more.
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/home/jovyan/HM-fashion-recs/models/autogluon_baseline"
Train Data Rows:    50000
Train Data Columns: 5
Label Column:       product_group_name
AutoGluon infers your prediction problem is: 'multiclass' (because dtype of label-column == object).
	First 10 (of 15) unique label values:  ['Garment Lower body', 'Nightwear', 'Swimwear', 'Garment Upper body', 'Under

## 4. Evaluate Results

In [13]:
# Model leaderboard
predictor.leaderboard(silent=True)

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,XGBoost,0.4112,accuracy,0.007834,1.968653,0.007834,1.968653,1,True,9
1,WeightedEnsemble_L2,0.4112,accuracy,0.009043,2.161125,0.001209,0.192473,2,True,12
2,LightGBMLarge,0.4088,accuracy,0.074239,4.288283,0.074239,4.288283,1,True,11
3,LightGBM,0.4076,accuracy,0.018146,1.859618,0.018146,1.859618,1,True,3
4,NeuralNetTorch,0.4064,accuracy,0.015325,39.384691,0.015325,39.384691,1,True,10
5,NeuralNetFastAI,0.4056,accuracy,0.033818,43.665605,0.033818,43.665605,1,True,1
6,LightGBMXT,0.4052,accuracy,0.104505,4.459575,0.104505,4.459575,1,True,2
7,CatBoost,0.4000,accuracy,0.004950,26.022491,0.004950,26.022491,1,True,6
8,RandomForestGini,0.3572,accuracy,0.176944,7.336424,0.176944,7.336424,1,True,4
9,RandomForestEntr,0.3560,accuracy,0.169338,8.512038,0.169338,8.512038,1,True,5


In [14]:
# Feature importance
predictor.feature_importance(train_ag)

Computing feature importance via permutation shuffling for 5 features using 5000 rows with 5 shuffle sets...
	0.92s	= Expected runtime (0.18s per shuffle set)
	0.46s	= Actual runtime (Completed 5 of 5 shuffle sets)


,importance,stddev,p_value,n,p99_high,p99_low
price,0.06112,0.011439,0.000141,5,0.084673,0.037567
sales_channel_id,0.01464,0.003251,0.000274,5,0.021334,0.007946
age,0.00140,0.001655,0.065784,5,0.004808,-0.002008
fashion_news_frequency,0.00108,0.001073,0.043823,5,0.003290,-0.001130
club_member_status,0.00004,0.000498,0.433097,5,0.001065,-0.000985


In [15]:
# Quick prediction example
sample_customer = train_data.drop(columns=[TARGET]).iloc[:5]
predictions = predictor.predict(sample_customer)
print("Predictions:")
print(predictions)

Predictions:
0    Garment Upper body
1    Garment Lower body
2    Garment Lower body
3    Garment Lower body
4    Garment Lower body
Name: product_group_name, dtype: object


## 5. Next Steps

To improve the model:
1. **More data**: Increase `SAMPLE_SIZE`
2. **Better features**: Add lag features from transaction history (as shown in class)
3. **Longer training**: Increase `time_limit` or use `best_quality` preset
4. **Different target**: Try predicting `department_name` or `index_group_name`
5. **Evaluation**: Use proper train/validation/test split by time